# Pollen · AI Cartoon

Turns the people and things in a real video into **cartoon versions of themselves**, not a filter: every frame is redrawn by an AI that follows the original pose and outlines.

| Style | Looks like |
|---|---|
| `toon` | flat 2D adult-cartoon: thick outlines, flat bold colour |
| `anime` | Japanese anime: cel shading, clean line art, big expressive eyes |
| `kungfu` | 3D animated feature film: soft golden light, stylised 3D characters |

**Free GPU, no money needed.**

**Kaggle** (recommended, about 30 free GPU hours a week):
1. Under *Settings*, set **Accelerator → GPU T4** and **Internet → On** (needs a verified phone number).
2. Under *Add Input → Upload*, add your video.

**Colab:** choose *Runtime → Change runtime type → T4 GPU*. You'll be asked to upload the video.

Then pick your options in the next cell and press **Run all**.

- **First run:** about 10 min, because the models need downloading.
- **After that:** about 2–5 min for every 4 seconds of video, per style.
- **The result:** an MP4. On Kaggle it's in the *Output* panel; on Colab it downloads automatically. Drop it into the Pollen studio to add particles or step inside it.

In [ ]:
# ---- your settings -----------------------------------------------------
STYLE    = "anime"     # "toon", "anime", "kungfu" or "all"
WHO      = "a person"  # describe the main subject, e.g. "a young man with short curly hair in a red hoodie"
START    = 0           # seconds into the video to begin
SECONDS  = 4           # how much to convert (longer = slower; 4-8 is a good start)
FPS      = 8           # drawings per second (8 looks hand-animated, 12 is smoother but slower)
SIZE     = 512         # long side in pixels while drawing (512 is the sweet spot on a free T4)
LIKENESS = 0.5         # 0 = wild cartoon, 1 = stays very close to the real footage
SMOOTH   = False       # True blends extra in-between frames (smoother, less "drawn")
SEED     = 7           # change for a different take
VIDEO_URL = ""         # optional: a direct link to a video instead of uploading

In [ ]:
# ---- install (about 1 min) ----------------------------------------------
!pip install -q "diffusers==0.42.1" "controlnet_aux==0.0.10" peft accelerate

In [ ]:
# ---- find the video and cut it into frames -------------------------------
import glob, os, shutil, subprocess, sys
from PIL import Image

IN_COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content" if IN_COLAB else os.getcwd()
EXTS = (".mp4", ".mov", ".webm", ".mkv", ".avi", ".m4v")

src = None
if VIDEO_URL:
    src = os.path.join(WORK, "input_video")
    subprocess.run(["wget", "-q", "-O", src, VIDEO_URL], check=True)
if not src:
    found = sorted(p for p in glob.glob("/kaggle/input/**/*", recursive=True) if p.lower().endswith(EXTS))
    src = found[0] if found else None
if not src and IN_COLAB:
    from google.colab import files
    src = os.path.abspath(next(iter(files.upload())))
assert src, "No video found. On Kaggle use Add Input -> Upload, or set VIDEO_URL."
print("video:", src)

FR = os.path.join(WORK, "frames")
shutil.rmtree(FR, ignore_errors=True); os.makedirs(FR)
# long side = SIZE, short side rounded to a multiple of 8 (phone rotation is applied first)
vf = f"fps={FPS},scale='if(gt(iw,ih),{SIZE},-8)':'if(gt(iw,ih),-8,{SIZE})':flags=lanczos"
cut = ["-ss", str(START), "-t", str(SECONDS), "-i", src]
subprocess.run(["ffmpeg", "-y", "-v", "error", *cut, "-vf", vf, f"{FR}/%04d.png"], check=True)
AUDIO = os.path.join(WORK, "audio.m4a")
has_audio = subprocess.run(["ffmpeg", "-y", "-v", "error", *cut, "-vn", "-c:a", "aac", AUDIO]).returncode == 0 \
    and os.path.exists(AUDIO) and os.path.getsize(AUDIO) > 1000

frames = [Image.open(p).convert("RGB") for p in sorted(glob.glob(f"{FR}/*.png"))]
W, H = frames[0].size
print(f"{len(frames)} frames at {W}x{H}, audio: {has_audio}")

In [ ]:
# ---- read pose and outlines from every frame -----------------------------
# Pose keeps the body, hands and face moving exactly like the real person;
# line art keeps the shapes of people and objects.
import torch, gc
from controlnet_aux import OpenposeDetector, LineartDetector

pose_det = OpenposeDetector.from_pretrained("lllyasviel/Annotators").to("cuda")
line_det = LineartDetector.from_pretrained("lllyasviel/Annotators").to("cuda")
pose, lines = [], []
for f in frames:
    pose.append(pose_det(f, detect_resolution=512, image_resolution=512,
                         include_hand=True, include_face=True).resize((W, H)))
    lines.append(line_det(f, detect_resolution=512, image_resolution=512).resize((W, H)))
del pose_det, line_det; gc.collect(); torch.cuda.empty_cache()

def strip(imgs, n=4):
    pick = [imgs[i * (len(imgs) - 1) // max(1, n - 1)] for i in range(n)]
    out = Image.new("RGB", (W * n, H))
    for i, im in enumerate(pick): out.paste(im, (i * W, 0))
    return out.resize((W * n // 2, H // 2))
display(strip(frames)); display(strip(pose)); display(strip(lines))

In [ ]:
# ---- the three styles ------------------------------------------------------
NEG = ("photo, photorealistic, realistic skin, worst quality, low quality, blurry, jpeg artifacts, "
       "deformed, extra fingers, extra limbs, bad anatomy, watermark, text, signature")
STYLES = {
    "toon": dict(
        base="frankjoshua/toonyou_beta6",
        prompt="2d adult animated tv cartoon, flat colors, thick black outlines, simple shapes, "
               "bold saturated colors, cel shaded, cartoon character, big round eyes, clean vector art",
        neg="3d render, shading gradients, painterly",
        strength=0.08, pose=0.75, line=0.55),
    "anime": dict(
        base="Meina/MeinaMix_V11",
        prompt="anime screencap, anime character, cel shading, clean crisp line art, detailed anime eyes, "
               "vibrant colors, beautiful painted background, anime film still, masterpiece",
        neg="3d, western cartoon",
        strength=0.0, pose=0.8, line=0.6),
    "kungfu": dict(
        base="stablediffusionapi/disney-pixar-cartoon",
        prompt="3d animated feature film still, stylized 3d cartoon character, big expressive eyes, "
               "soft cinematic lighting, warm golden hour light, rich detailed environment, octane render",
        neg="2d, flat, sketch, anime",
        strength=0.03, pose=0.75, line=0.5),
}

In [ ]:
# ---- load the shared AI parts once ---------------------------------------
from diffusers import (AnimateDiffVideoToVideoControlNetPipeline, AutoencoderKL,
                       ControlNetModel, LCMScheduler, MotionAdapter)
DT = torch.float16
adapter = MotionAdapter.from_pretrained("wangfuyun/AnimateLCM", torch_dtype=DT, variant="fp16")
controlnets = [ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_openpose", torch_dtype=DT),
               ControlNetModel.from_pretrained("lllyasviel/control_v11p_sd15_lineart", torch_dtype=DT)]
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse", torch_dtype=DT)

def load_style(key):
    s = STYLES[key]
    pipe = AnimateDiffVideoToVideoControlNetPipeline.from_pretrained(
        s["base"], motion_adapter=adapter, controlnet=controlnets, vae=vae, torch_dtype=DT)
    # AnimateLCM: about 8 steps instead of 25+, roughly 3x faster
    pipe.scheduler = LCMScheduler.from_config(pipe.scheduler.config, beta_schedule="linear")
    pipe.load_lora_weights("wangfuyun/AnimateLCM", weight_name="AnimateLCM_sd15_t2v_lora.safetensors",
                           adapter_name="lcm")
    pipe.set_adapters(["lcm"], [0.8])
    pipe.vae.enable_slicing()
    return pipe.to("cuda")

In [ ]:
# ---- draw ------------------------------------------------------------------
# The clip is drawn 16 frames at a time; chunks overlap by 4 and are blended,
# which keeps memory inside a free 16 GB GPU on any clip length.
import numpy as np, time

CHUNK, OVER = 16, 4

def draw(key):
    s = STYLES[key]
    pipe = load_style(key)
    strength = float(np.clip(0.85 - 0.35 * LIKENESS + s["strength"], 0.35, 0.95))
    n = len(frames)
    starts = list(range(0, max(1, n - OVER), CHUNK - OVER))
    acc = np.zeros((n, H, W, 3), np.float32); wsum = np.zeros((n, 1, 1, 1), np.float32)
    t0 = time.time()
    for k, a in enumerate(starts):
        b = min(n, a + CHUNK); a = max(0, b - CHUNK)  # last chunk is always full length
        with torch.inference_mode():
            out = pipe(
                video=frames[a:b], prompt=f"{s['prompt']}, {WHO}", negative_prompt=f"{NEG}, {s['neg']}",
                height=H, width=W, num_inference_steps=12, guidance_scale=2.0, strength=strength,
                conditioning_frames=[pose[a:b], lines[a:b]],
                controlnet_conditioning_scale=[s["pose"], s["line"]],
                generator=torch.Generator("cuda").manual_seed(SEED), decode_chunk_size=4,
            ).frames[0]
        # triangular weights fade each chunk in and out across the overlap
        m = b - a
        ramp = np.minimum(np.arange(1, m + 1), np.arange(m, 0, -1)).astype(np.float32)
        ramp = np.minimum(ramp / OVER, 1.0)
        if a == 0: ramp[:OVER] = 1
        if b == n: ramp[-OVER:] = 1
        for i, im in enumerate(out):
            acc[a + i] += ramp[i] * np.asarray(im.resize((W, H)), np.float32)
            wsum[a + i] += ramp[i]
        print(f"  {key}: frames {a}-{b} done ({time.time() - t0:.0f}s)")
        if b == n: break
    del pipe; gc.collect(); torch.cuda.empty_cache()
    return [Image.fromarray((acc[i] / wsum[i]).clip(0, 255).astype(np.uint8)) for i in range(n)]

def save(key, drawn):
    d = os.path.join(WORK, f"out_{key}"); shutil.rmtree(d, ignore_errors=True); os.makedirs(d)
    for i, im in enumerate(drawn): im.save(f"{d}/{i:04d}.png")
    motion = "minterpolate=fps=24:mi_mode=mci" if SMOOTH else "fps=24"
    vf = f"{motion},scale=iw*2:ih*2:flags=lanczos,format=yuv420p"
    out = os.path.join(WORK, f"pollen_{key}.mp4")
    aud = ["-i", AUDIO, "-c:a", "aac", "-shortest"] if has_audio else []
    subprocess.run(["ffmpeg", "-y", "-v", "error", "-framerate", str(FPS), "-i", f"{d}/%04d.png", *aud,
                    "-vf", vf, "-c:v", "libx264", "-crf", "17", "-movflags", "+faststart", out], check=True)
    # side-by-side: real footage | cartoon
    cmp_out = os.path.join(WORK, f"compare_{key}.mp4")
    subprocess.run(["ffmpeg", "-y", "-v", "error", "-framerate", str(FPS), "-i", f"{FR}/%04d.png",
                    "-framerate", str(FPS), "-i", f"{d}/%04d.png",
                    "-filter_complex", "[0][1]hstack,fps=24,format=yuv420p", "-c:v", "libx264", "-crf", "20", cmp_out],
                   check=True)
    return out, cmp_out

results = []
for key in (list(STYLES) if STYLE == "all" else [STYLE]):
    print("drawing", key)
    results += save(key, draw(key))
print("done:", *results, sep="\n  ")

In [ ]:
# ---- watch and download --------------------------------------------------
from base64 import b64encode
from IPython.display import HTML
for p in results:
    data = b64encode(open(p, "rb").read()).decode()
    display(HTML(f"<p>{os.path.basename(p)}</p><video controls loop autoplay muted width=640 "
                 f"src='data:video/mp4;base64,{data}'></video>"))
if IN_COLAB:
    from google.colab import files
    for p in results: files.download(p)
else:
    print("Kaggle: open the Output panel on the right (or /kaggle/working) to download the MP4s.")